[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/assignments/week03_assignment.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 3 Assignment: Sample Size & MDE Estimation

## QuickCart — Planning the Checkout Experiment

QuickCart is preparing to test a **new checkout flow** that simplifies the payment process from 4 steps to 2. The product team believes this will increase the conversion rate (proportion of cart-starters who complete a purchase).

Before launching, the data science team needs to answer:
- **How many users** do we need in each group?
- **How long** will the test need to run given our traffic?
- What is the **smallest effect** we can reliably detect?

This assignment builds the sample size estimation machinery from scratch.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

## Historical Data

Below is a simulated dataset of QuickCart’s historical checkout metrics. Each row represents one user session that reached the cart page.

In [ ]:
np.random.seed(123)

n_sessions = 100_000

# Conversion rate (Bernoulli): ~12% baseline
conversion = np.random.binomial(1, 0.12, n_sessions)

# Revenue per converting session (log-normal, heavy tail)
revenue = np.where(
    conversion == 1,
    np.random.lognormal(mean=3.5, sigma=0.8, size=n_sessions),
    0
)

# Average order value for converters
aov = np.where(conversion == 1, revenue, np.nan)

df_hist = pd.DataFrame({
    'converted': conversion,
    'revenue': np.round(revenue, 2),
    'aov': np.round(aov, 2)
})

print(f"Sessions: {len(df_hist):,}")
print(f"Conversion rate: {df_hist['converted'].mean():.4f}")
print(f"Mean revenue (all sessions): ${df_hist['revenue'].mean():.2f}")
print(f"Std revenue (all sessions): ${df_hist['revenue'].std():.2f}")
print(f"Mean AOV (converters only): ${df_hist['aov'].dropna().mean():.2f}")
df_hist.describe()

---

## Task 1: Implement `get_sample_size`

### Background

The classic sample size formula for a two-sample z-test (equal groups) is:

$$n = \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot 2\sigma^2}{\delta^2}$$

where:
- $z_{\alpha/2}$: critical value for the significance level (e.g., 1.96 for $\alpha=0.05$)
- $z_{\beta}$: critical value for the power (e.g., 0.84 for $\beta=0.20$, giving 80% power)
- $\sigma$: standard deviation of the metric
- $\delta = \mu \cdot (\text{effect} - 1)$: absolute difference to detect (where effect is the multiplicative factor, e.g., 1.05 for a 5% lift)

### Problem

Implement a function that computes the required sample size **per group**.

<details>
<summary>Hint 1: Computing z-values (click to expand)</summary>

```python
z_alpha = stats.norm.ppf(1 - alpha / 2)
z_beta = stats.norm.ppf(1 - beta)
```

</details>

<details>
<summary>Hint 2: Computing delta</summary>

If `eff = 1.05` and `mu = 100`, then the absolute effect is `delta = mu * eff - mu = mu * (eff - 1) = 5`.

</details>

<details>
<summary>Hint 3: Ceiling the result</summary>

Always round up to the next integer: `int(np.ceil(n))`

</details>

In [ ]:
def get_sample_size(mu, std, eff, alpha=0.05, beta=0.2):
    """Compute required sample size per group for a two-sample test.
    
    Parameters
    ----------
    mu : float
        Baseline mean of the metric.
    std : float
        Standard deviation of the metric.
    eff : float
        Expected effect as a multiplicative factor.
        e.g., 1.05 means a 5% increase, 0.95 means a 5% decrease.
    alpha : float
        Significance level (default 0.05, two-sided).
    beta : float
        Type II error rate (default 0.2, i.e., 80% power).
    
    Returns
    -------
    int
        Required sample size per group (rounded up).
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Test Cases for Task 1 ---

# Test 1: Known scenario
# mu=100, std=50, 5% effect -> delta=5
# n = (1.96 + 0.84)^2 * 2 * 50^2 / 5^2 = 7.84 * 5000 / 25 = 1568.0
n_test = get_sample_size(mu=100, std=50, eff=1.05)
print(f"Test 1: n = {n_test} (expected: 1568 or 1569)")
assert 1560 <= n_test <= 1575, f"Expected ~1568, got {n_test}"

# Test 2: Larger effect needs fewer samples
n_large_effect = get_sample_size(mu=100, std=50, eff=1.10)
print(f"Test 2: n (10% effect) = {n_large_effect}")
assert n_large_effect < n_test, "Larger effect should need fewer samples"

# Test 3: Higher variance needs more samples
n_high_var = get_sample_size(mu=100, std=100, eff=1.05)
print(f"Test 3: n (high variance) = {n_high_var}")
assert n_high_var > n_test, "Higher variance should need more samples"

# Test 4: Stricter alpha needs more samples
n_strict = get_sample_size(mu=100, std=50, eff=1.05, alpha=0.01)
print(f"Test 4: n (alpha=0.01) = {n_strict}")
assert n_strict > n_test, "Stricter alpha should need more samples"

# Test 5: Higher power (lower beta) needs more samples
n_high_power = get_sample_size(mu=100, std=50, eff=1.05, beta=0.1)
print(f"Test 5: n (90% power) = {n_high_power}")
assert n_high_power > n_test, "Higher power should need more samples"

print("\nAll Task 1 tests passed!")

---

## Task 2: Implement `estimate_sample_size`

### Problem

Now wrap your `get_sample_size` function to work directly with a DataFrame. This function should:
1. Extract the mean and standard deviation from the specified metric column
2. Compute sample sizes for a list of effect sizes
3. Return a clean DataFrame with the results

<details>
<summary>Hint: Structure</summary>

```python
mu = df[metric_name].mean()
std = df[metric_name].std()
results = []
for eff in effects:
    n = get_sample_size(mu, std, eff, alpha, beta)
    results.append({'effect': eff, 'sample_size': n})
return pd.DataFrame(results)
```

</details>

In [ ]:
def estimate_sample_size(df, metric_name, effects, alpha=0.05, beta=0.2):
    """Estimate required sample sizes for a list of effect sizes.
    
    Parameters
    ----------
    df : pd.DataFrame
        Historical data containing the metric.
    metric_name : str
        Column name of the metric to analyze.
    effects : list of float
        List of multiplicative effects to evaluate.
        e.g., [1.01, 1.03, 1.05, 1.10] for 1%, 3%, 5%, 10% lifts.
    alpha : float
        Significance level (default 0.05).
    beta : float
        Type II error rate (default 0.2).
    
    Returns
    -------
    pd.DataFrame
        Columns: ['effect', 'sample_size']
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Test Cases for Task 2 ---

effects_list = [1.01, 1.03, 1.05, 1.10, 1.20]

result_revenue = estimate_sample_size(df_hist, 'revenue', effects_list)
print("Sample sizes for 'revenue' metric:")
print(result_revenue.to_string(index=False))
print()

# Sanity checks
assert isinstance(result_revenue, pd.DataFrame), "Should return a DataFrame"
assert list(result_revenue.columns) == ['effect', 'sample_size'], f"Unexpected columns: {result_revenue.columns.tolist()}"
assert len(result_revenue) == len(effects_list), "Should have one row per effect"

# Sample sizes should decrease as effect increases
sizes = result_revenue['sample_size'].tolist()
assert sizes == sorted(sizes, reverse=True), "Larger effects should need fewer samples"

# Check conversion metric too
result_conv = estimate_sample_size(df_hist, 'converted', effects_list)
print("Sample sizes for 'converted' metric:")
print(result_conv.to_string(index=False))

print("\nAll Task 2 tests passed!")

---

## Task 3: Test Duration Planning

### Problem

QuickCart gets **50,000 unique users per day** reaching the checkout page. The experiment will split traffic 50/50 (25,000 per group per day).

Using your `estimate_sample_size` results for the **revenue** metric:
1. Compute how many **days** each MDE would require
2. Also express this in **weeks** (rounded up)
3. Create a summary table and a plot showing MDE vs test duration

The team has a maximum test duration budget of **4 weeks**. Which is the smallest effect size (MDE) that QuickCart can reliably detect within this budget?

<details>
<summary>Hint: Duration calculation</summary>

```python
daily_traffic_per_group = 50000 / 2  # 50/50 split
days_needed = np.ceil(sample_size / daily_traffic_per_group)
weeks_needed = np.ceil(days_needed / 7)
```

</details>

In [ ]:
# YOUR CODE HERE
# 1. Use estimate_sample_size with the revenue metric and a range of effects
#    Suggested effects: [1.01, 1.02, 1.03, 1.05, 1.07, 1.10, 1.15, 1.20]
# 2. Compute days and weeks required for each MDE
# 3. Create a table and a plot
# 4. Identify the MDE achievable within 4 weeks

daily_traffic = 50_000  # total daily users
max_weeks = 4

**Your answer:**

*What is the smallest MDE detectable within 4 weeks? What does this mean for the product team’s expectations?*

---

## Task 4: Impact of Variance Reduction

### Problem

The revenue metric has high variance due to outliers (bulk orders). One common technique is **outlier capping** (winsorization) — capping values at a certain percentile.

Suppose capping at the 99th percentile reduces the standard deviation by approximately **30%**.

1. Recompute sample sizes with `std_new = std_original * 0.7`
2. Create a side-by-side comparison table: original vs reduced variance
3. Plot both curves (MDE vs sample size) on the same chart
4. By what factor does variance reduction shrink the required sample size?

<details>
<summary>Hint: The relationship</summary>

Since $n \propto \sigma^2$, reducing $\sigma$ by 30% reduces $n$ by a factor of $0.7^2 = 0.49$, i.e., roughly halves the required sample size.

Verify this empirically with your function.

</details>

In [ ]:
# YOUR CODE HERE
# 1. Compute sample sizes with original std
# 2. Compute sample sizes with std * 0.7
# 3. Create comparison table
# 4. Plot both curves
# 5. Compute the reduction factor

**Your analysis:**

*How much does the 30% variance reduction help? What is the new MDE achievable within 4 weeks? Would you recommend implementing outlier capping for QuickCart’s experimentation platform?*

---

## Bonus: Sensitivity Plot

Create a **heatmap** showing required test duration (in weeks) as a function of:
- X-axis: MDE (1% to 20%)
- Y-axis: Power (70%, 80%, 90%, 95%)

This gives the product team a single visual to understand the tradeoffs.

In [ ]:
# YOUR CODE HERE (optional)